# ฮองเรียน — เซิร์ฟเวอร์แปลซับบน Colab GPU (A100)

รันโมเดลแปลภาษาไทยตัวใหญ่บน Colab แล้วให้เซิร์ฟเวอร์ในเครื่องคุณ (`start-server.cmd`) ส่งงานแปลมาที่นี่
ถ้าโน้ตบุ๊กนี้ปิดหรือหลุด เครื่องคุณจะกลับไปแปลด้วย `agy` เองโดยอัตโนมัติ ไม่มีอะไรค้าง

**วิธีใช้**
1. Runtime → Change runtime type → **A100 GPU** แล้วกด **Runtime → Run all**
2. รอจนเซลล์ที่ 4 พิมพ์ 3 บรรทัด `UDTH_LLM_...` ออกมา (ครั้งแรกใช้เวลาดาวน์โหลดโมเดล 12B ราว 5-10 นาที)
3. คัดลอก 3 บรรทัดนั้นไปวางท้ายไฟล์ `.env` ของโปรเจกต์ แล้วปิด-เปิด `start-server.cmd` ใหม่
4. เปิดแท็บนี้ค้างไว้ระหว่างเรียน (เซลล์สุดท้ายรันค้างไว้เพื่อให้ Colab รู้ว่ายังใช้งานอยู่)

> ข้อตกลงของ Colab อนุญาตให้เปิดบริการแบบนี้เฉพาะแพ็กเกจเสียเงินที่ยังมี compute units เหลือ
> ([Colab FAQ](https://research.google.com/colaboratory/faq.html)) — ถ้า units หมด ให้หยุดโน้ตบุ๊กนี้

> URL ที่ได้เปลี่ยนทุกครั้งที่รันใหม่ ต้องอัปเดต `.env` ทุกครั้ง · API key สุ่มใหม่ทุกครั้ง ห้ามแชร์

## 1. เลือกโมเดล

In [ ]:
# ค่าเริ่มต้น: Typhoon 2.1 (Gemma 3) 12B — ภาษาไทยดี เร็ว ใช้ GPU ราว 24 GB
MODEL = "scb10x/typhoon2.1-gemma3-12b"
QUANTIZATION = None
MAX_MODEL_LEN = 32768

# แปลดีที่สุด แต่ดาวน์โหลดราว 140 GB และต้องใช้ A100 80GB:
# Typhoon 2 70B ย่อเป็น FP8 (A100 รันแบบ weight-only ผ่าน Marlin) ใช้ GPU ราว 70 GB
# MODEL = "scb10x/llama3.1-typhoon2-70b-instruct"
# QUANTIZATION = "fp8"
# MAX_MODEL_LEN = 16384

PORT = 8000

## 2. ติดตั้ง vLLM และ cloudflared

ถ้าโมเดลที่เลือกต้องยอมรับเงื่อนไขก่อนดาวน์โหลด (Hugging Face ตอบ 401/403):
เปิดหน้าโมเดลบน huggingface.co กดยอมรับเงื่อนไข แล้วใส่ token ไว้ใน Colab **Secrets** ชื่อ `HF_TOKEN`

In [ ]:
!pip install -q vllm
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. เปิดเซิร์ฟเวอร์โมเดล (OpenAI-compatible) พร้อม API key

In [ ]:
import os, secrets, subprocess, time, urllib.request, json

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

API_KEY = secrets.token_urlsafe(32)
command = ["vllm", "serve", MODEL, "--api-key", API_KEY, "--port", str(PORT),
           "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92"]
if QUANTIZATION:
    command += ["--quantization", QUANTIZATION]
log = open("/content/vllm.log", "w")
server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)

def ready():
    request = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models",
                                     headers={"Authorization": f"Bearer {API_KEY}"})
    try:
        return urllib.request.urlopen(request, timeout=5).status == 200
    except Exception:
        return False

started = time.time()
while not ready():
    if server.poll() is not None:
        print(open("/content/vllm.log").read()[-4000:])
        raise RuntimeError("vLLM หยุดทำงาน ดู log ด้านบน")
    time.sleep(10)
    print(f"  กำลังโหลดโมเดล ... {int(time.time() - started)} วินาที", end="\r")
print(f"\nโมเดลพร้อมใน {int(time.time() - started)} วินาที")

## 4. เปิด tunnel แล้วคัดลอก 3 บรรทัดนี้ไปใส่ `.env`

In [ ]:
import re

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        public = found.group(0)
        break

print("วางท้ายไฟล์ .env แล้วปิด-เปิด start-server.cmd ใหม่:\n")
print(f"UDTH_LLM_URL={public}/v1")
print(f"UDTH_LLM_KEY={API_KEY}")
print(f"UDTH_LLM_MODEL={MODEL}")

## 5. ทดสอบแปลผ่าน tunnel

In [ ]:
time.sleep(5)  # the tunnel needs a moment before it routes
body = {"model": MODEL, "temperature": 0.2, "max_tokens": 200, "messages": [{"role": "user", "content":
        "Translate into natural Thai for a programming lecture subtitle: "
        "So the function takes a name and returns a greeting string."}]}
request = urllib.request.Request(f"{public}/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
started = time.time()
answer = json.load(urllib.request.urlopen(request, timeout=120))["choices"][0]["message"]["content"]
print(f"{time.time() - started:.1f}s  {answer}")

## 6. รันค้างไว้ระหว่างเรียน (กด Stop เมื่อเลิกใช้)

In [ ]:
while True:
    alive = server.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "พร้อมใช้งาน" if alive and ready() else "มีปัญหา — ดู /content/vllm.log", end="\r")
    if not alive:
        break
    time.sleep(60)